# 06 · Zero-shot and adaptation gain: SegFormer MiT-B3

How well does a model trained on Japanese data work on the local Cirebon and Bali images, and how much does adding local data help?

| Model | Training data | Evaluated on |
|---|---|---|
| **A (zero-shot)** | Japanese training set only (v2022 + BePLi, 3,956 images) | all 150 local images, sliced per configuration in the analysis |
| **B (adapted)** | Japanese training set + local `L_train`; 10% of batch slots hold a local image, same steps per epoch as A | `L_test` of that configuration |

- **Local splits:** 3 random 50/50 splits stratified by site (Cirebon, Bali), each used in both directions (half 1 → half 2 and back), giving 6 configurations. Every local image is a test image exactly once per split.
- **Seeds:** 42, 43, 44. A and B with the same seed are paired.
- **Epoch selection:** both A and B keep the epoch with the best Japanese validation mIoU. Local test data is never used to choose anything.
- **Adaptation gain:** Δ = B − A per local image (averaged over the configurations in which the image is a test image, then over seeds), with a 95% CI from a paired bootstrap (2,000 resamples).
- **Metrics on local data:** per-class IoU, **mIoU litter** (mean IoU of artificial and natural, without background), 3-class mIoU, median per-image IoU, precision, recall, Dice, area error (MAE per class and for total litter), bias, correlation, and inference time.

**Training:** AdamW, learning rate 1.206e-4, weight decay 1.040e-2 (best trial of notebook 04, validation mIoU 0.7516), poly schedule (power 1.0); 12 epochs, batch 8, 512 crop, cross-entropy + Dice loss, same augmentation as the Japanese training notebooks.

**Sessions:** 21 runs in total (3 A + 18 B, about 45–55 min each). Seeds can be split across sessions or accounts with `TRAIN_SEEDS`. A previous output folder with the same name as `OUT` is found automatically under `/kaggle/input` (`AUTO_RESUME`); finished runs are skipped.

**Probabilities for the ensemble test:** B runs with a seed in `SAVE_PROBA_SEEDS` (default 44) save per-pixel artificial and natural probabilities on `L_test` to `proba/B_s44_c*.npz` (uint8). Notebook 07 uses them.

**Kaggle inputs:** the `beach-litter` and `databepliv2` datasets and the output of notebook 02. GPU and Internet on.

## 1. Packages

In [ ]:
%pip -q install 'segmentation-models-pytorch==0.5.0' 'albumentations==2.0.8'

## 2. Configuration
The SegFormer hyperparameters are written in directly from the best tuning trial of notebook 04 (trial 1, validation mIoU 0.7516), so the tuning output is not needed as input. Every A and B run uses the same values.

In [ ]:
import hashlib
import json
import math
import random
import re
import shutil
import time
import warnings
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
import torch.nn.functional as F
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset, Sampler
from tqdm.auto import tqdm

SESSION_START = time.time()
warnings.filterwarnings("ignore", category=RuntimeWarning)

STAGES = ("prepare", "train", "analyze", "final")
TRAIN_SEEDS = (42, 43, 44)
RESUME_DIRS = []
AUTO_RESUME = True
SAVE_PROBA_SEEDS = (44,)
RUN_FINAL_MODEL = True

TUNED_PARAMS = {
    "learning_rate": 1.2057126278744777e-4,
    "weight_decay": 1.0401663679887312e-2,
}
TUNING_BEST_TRIAL = 1
TUNING_BEST_VAL_MIOU = 0.7516000835153602
print(f"SegFormer hyperparameters (tuning trial {TUNING_BEST_TRIAL}): {TUNED_PARAMS}")
print(f"Best tuning validation mIoU: {TUNING_BEST_VAL_MIOU:.4f}")

V2022_IMG = Path("/kaggle/input/datasets/rafiadabhi/beach-litter/image")
V2022_MASK = Path("/kaggle/input/datasets/rafiadabhi/beach-litter/mask")
LOCAL_DIR = Path("/kaggle/input/datasets/rafiadabhi/beach-litter/data_lokal_cirebonbali/DATA LOKAL")
LOCAL_IMG, LOCAL_MASK = LOCAL_DIR / "images", LOCAL_DIR / "mask"
BEPLI_DIR = Path("/kaggle/input/datasets/rafiadabhi/databepliv2")
BEPLI_LIST = BEPLI_DIR / "pembagian_anotasi_bepli.md"
PSEUDO_RUN_DIR = Path("/kaggle/input/notebooks/rafiadabhi/pseudo-mask2former-checkpoint")
BEPLI_MASK_DIR = PSEUDO_RUN_DIR / "bepli_pseudo_masks"
V2022_SPLIT_CSV = PSEUDO_RUN_DIR / "v2022_split.csv"
FINAL_RUN_DIR = None
INPUT_ROOT = Path("/kaggle/input")

OUT = Path("/kaggle/working/adaptation_segformer_b3")
CACHE = Path("/tmp/cache_512")

EXPECTED_LOCAL = {"bali": 61, "cirebon": 89}
EXCLUDE_LOCAL = ()
MASK_NAME_FIX = {"TT10": "TT110"}
AMBIGUOUS_LIMIT = 0.005
SPLIT_SEED = 123
N_LOCAL_SPLITS = 3
P_LOCAL = 0.1
N_BOOT = 2000

SEED = 42
CLASSES = ["background", "artificial", "natural"]
TAGS = {1: "art", 2: "nat"}
PALETTE = np.array([[0, 0, 0], [255, 0, 0], [0, 255, 0]], dtype=np.int32)
IGNORE = 255
SHORT_SIDE, CROP, BATCH_SIZE, NUM_WORKERS = 512, 512, 8, 4
ENCODER, ENCODER_WEIGHTS = "mit_b3", "imagenet"
POLY_POWER, EPOCHS = 1.0, 12
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
OBJECTIVE_SOURCES = ("v2022", "bepli")

START_JOB_LIMIT_H = 11.0
HARD_STOP_H = 11.5
DEFAULT_JOB_H = 0.9

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", ("GPU is off. Open Settings > Accelerator, choose GPU T4 x1 or P100, "
                               "then rerun.")
USE_AMP = DEVICE.type == "cuda"

OUT.mkdir(parents=True, exist_ok=True)
for sub in ("eval", "runs", "history", "splits", "figures", "proba"):
    (OUT / sub).mkdir(exist_ok=True)
if not RESUME_DIRS and AUTO_RESUME:
    found = sorted({p.parent.parent for p in INPUT_ROOT.rglob("runs/*.json") if p.parent.parent.name == OUT.name})
    for d in found:
        n_runs = len(list((d / "runs").glob("*.json")))
        print(f"Previous output found: {d} ({n_runs} runs)")
    assert len(found) <= 1, ("More than one previous output folder found; set RESUME_DIRS to the latest one only.")
    RESUME_DIRS = [str(d) for d in found]
    if not found:
        print("No previous output in the inputs; starting from scratch.")
for d in RESUME_DIRS:
    shutil.copytree(d, OUT, dirs_exist_ok=True)
    print("Resumed from:", d)


def hours() -> float:
    return (time.time() - SESSION_START) / 3600


def log(msg: str) -> None:
    print(f"[{time.strftime('%H:%M:%S')} | {hours():.2f} h] {msg}", flush=True)


def seed_all(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_all(SEED)
log(f"Device: {DEVICE} {torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else ''} | training seeds: {TRAIN_SEEDS}")

## 3. Japanese data
Same as the Japanese training notebooks: the v2022 split from `v2022_split.csv` and a 70/10/20 BePLi split with a fixed seed. If `FINAL_RUN_DIR` is set, the BePLi split is checked against that notebook's `bepli_split.csv`.

In [ ]:
def resolve_dir(configured: Path, required: list[str], label: str) -> Path:
    missing = [n for n in required if not (configured / n).is_file()]
    if not missing:
        return configured
    for p in sorted(INPUT_ROOT.rglob(required[0])):
        if all((p.parent / n).is_file() for n in required):
            log(f"{label}: incomplete at {configured}, using {p.parent}")
            return p.parent
    raise FileNotFoundError(f"{label}: {len(missing)} files missing in {configured}, e.g. {missing[:3]}")


def read_bepli_list(md_path: Path) -> list[str]:
    blocks = re.findall(r"```(.*?)```", md_path.read_text(encoding="utf-8"), re.S)
    return sorted({n for block in blocks for n in re.findall(r"\b\d{6}\b", block)})


def index_pngs(root: Path, preferred: str = "original_images") -> dict:
    index = {}
    for p in sorted(root.rglob("*.png")):
        if p.name not in index or preferred in p.parts:
            index[p.name] = p
    return index


v_split = pd.read_csv(V2022_SPLIT_CSV)
V2022_IMG = resolve_dir(V2022_IMG, v_split["image"].tolist(), "v2022 images")
V2022_MASK = resolve_dir(V2022_MASK, v_split["mask"].tolist(), "v2022 masks")
v2022 = pd.DataFrame({
    "id": "v2022_" + v_split["image"].str.replace(r"\.\w+$", "", regex=True),
    "source": "v2022",
    "image_path": [str(V2022_IMG / f) for f in v_split["image"]],
    "mask_path": [str(V2022_MASK / f) for f in v_split["mask"]],
    "mask_format": "rgb",
    "split": v_split["split"],
})

numbers = read_bepli_list(BEPLI_LIST)
png_index = index_pngs(BEPLI_DIR)
assert all(f"{n}.png" in png_index for n in numbers), "Some BePLi images are missing"
assert all((BEPLI_MASK_DIR / f"{n}.png").is_file() for n in numbers), "Some BePLi pseudo-label masks are missing"
bepli = pd.DataFrame({
    "id": [f"bepli_{n}" for n in numbers], "source": "bepli",
    "image_path": [str(png_index[f"{n}.png"]) for n in numbers],
    "mask_path": [str(BEPLI_MASK_DIR / f"{n}.png") for n in numbers], "mask_format": "index",
})
natural_pct = [float((cv2.resize(cv2.imread(p, cv2.IMREAD_GRAYSCALE), (128, 128),
                                 interpolation=cv2.INTER_NEAREST) == 2).mean() * 100)
               for p in tqdm(bepli["mask_path"], desc="BePLi natural area")]
strata = pd.qcut(pd.Series(natural_pct).rank(method="first"), 4, labels=False)
b_train, b_rest = train_test_split(bepli.index, test_size=0.30, stratify=strata, random_state=SEED)
b_val, b_test = train_test_split(b_rest, test_size=2 / 3, stratify=strata[b_rest], random_state=SEED)
bepli["split"] = "train"
bepli.loc[b_val, "split"] = "val"
bepli.loc[b_test, "split"] = "test"

if FINAL_RUN_DIR is not None and (Path(FINAL_RUN_DIR) / "bepli_split.csv").is_file():
    ref = pd.read_csv(Path(FINAL_RUN_DIR) / "bepli_split.csv")
    ours = pd.DataFrame({"image": bepli["image_path"].map(lambda p: Path(p).name), "split": bepli["split"]})
    assert ref[["image", "split"]].reset_index(drop=True).equals(ours.reset_index(drop=True)), \
        "BePLi split differs from the final notebook"
    log("Japanese split identical to the final notebook.")
else:
    log("FINAL_RUN_DIR not set; BePLi split not cross-checked (deterministic with seed 42).")

japan = pd.concat([v2022, bepli], ignore_index=True)
train_df = japan[japan.split == "train"].reset_index(drop=True)
val_df = japan[japan.split == "val"].reset_index(drop=True)
STEPS_PER_EPOCH = len(train_df) // BATCH_SIZE
print(pd.crosstab(japan["source"], japan["split"], margins=True))
print(f"Steps per epoch (A and B): {STEPS_PER_EPOCH}")

## 4. Local data
Image `<name>.png` pairs with mask `<name>_mask.png`. RGBA masks: red → 1 (artificial), green → 2 (natural), other bright pixels → 255 (ignored), the rest → background. Transparent pixels (alpha < 128) are background. Masks are resized to the image size with nearest-neighbour interpolation. `MASK_NAME_FIX` pairs the mask saved as `TT10` with image `TT110`.

In [ ]:
def index_dir(directory: Path, strip_mask_suffix: bool) -> dict:
    index = {}
    for path in sorted(directory.iterdir()):
        if path.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
            continue
        stem = re.sub(r"_mask(?:\s*\d+)?$", "", path.stem, flags=re.I) if strip_mask_suffix else path.stem
        if stem in index:
            raise ValueError(f"Duplicate base name: {index[stem].name} and {path.name}")
        index[stem] = path
    return index


def read_local_mask(path: str) -> tuple[np.ndarray, float]:
    with Image.open(path) as im:
        if im.mode in ("1", "L", "I", "I;16", "F"):
            raise ValueError(f"{Path(path).name}: {im.mode} mask has no class colours")
        arr = np.asarray(im if im.mode == "RGBA" else im.convert("RGB"))
    visible = arr[..., 3] >= 128 if arr.shape[2] == 4 else np.ones(arr.shape[:2], bool)
    rgb = arr[..., :3].astype(np.int16)
    r, g, b = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    red = visible & (r >= 128) & (r >= g + 25) & (r >= b + 25)
    green = visible & (g >= 128) & (g >= r + 25) & (g >= b + 25)
    ambiguous = visible & (rgb.max(axis=2) >= 128) & ~(red | green)
    labels = np.zeros(arr.shape[:2], np.uint8)
    labels[red], labels[green], labels[ambiguous] = 1, 2, IGNORE
    return labels, float(ambiguous.mean())


images, masks = index_dir(LOCAL_IMG, False), index_dir(LOCAL_MASK, True)
for wrong, right in MASK_NAME_FIX.items():
    if wrong in masks and right not in masks:
        masks[right] = masks.pop(wrong)
        print(f"Mask {masks[right].name} paired with image {right}")
no_mask, no_image = sorted(images.keys() - masks.keys()), sorted(masks.keys() - images.keys())
assert not no_mask and not no_image, f"No mask: {no_mask[:10]} | no image: {no_image[:10]}"

local = pd.DataFrame({"stem": sorted(images)})
local = local[~local.stem.isin(EXCLUDE_LOCAL)].reset_index(drop=True)
local["source"] = np.where(local.stem.str.fullmatch(r"BALI\d+", case=False), "bali", "cirebon")
local["id"] = "local_" + local.stem
local["image_path"] = local.stem.map(lambda s: str(images[s]))
local["mask_path"] = local.stem.map(lambda s: str(masks[s]))
local["mask_format"] = "local"
counts = local.source.value_counts().to_dict()
assert counts == {k: v for k, v in EXPECTED_LOCAL.items()}, f"Unexpected local counts: {counts}"
SOURCE_OF = dict(zip(local.id, local.source))
print(local.source.value_counts().to_string())

## 5. Local splits: 3 splits × 2 directions = 6 configurations
`split_hash` must be the same in notebooks 05 and 06.

In [ ]:
def split_local(frame: pd.DataFrame) -> dict:
    configs = {}
    for k in range(N_LOCAL_SPLITS):
        rng = np.random.default_rng(SPLIT_SEED + k)
        half_a, half_b = [], []
        for _, group in frame.groupby("source"):
            ids = sorted(group.id)
            order = rng.permutation(len(ids))
            cut = len(ids) // 2
            half_a += [ids[i] for i in order[:cut]]
            half_b += [ids[i] for i in order[cut:]]
        configs[2 * k] = {"train": sorted(half_a), "test": sorted(half_b)}
        configs[2 * k + 1] = {"train": sorted(half_b), "test": sorted(half_a)}
    return configs


LOCAL_CONFIGS = split_local(local)
for cfg in LOCAL_CONFIGS.values():
    assert not set(cfg["train"]) & set(cfg["test"])
    assert set(cfg["train"]) | set(cfg["test"]) == set(local.id)

split_payload = json.dumps({str(k): v for k, v in LOCAL_CONFIGS.items()}, sort_keys=True)
SPLIT_HASH = hashlib.sha256(split_payload.encode()).hexdigest()[:12]
split_file = OUT / "splits" / "local_configs.json"
if split_file.is_file():
    assert json.loads(split_file.read_text()) == json.loads(split_payload), \
        "Local splits differ from the previous session; do not merge the results."
split_file.write_text(split_payload)
for k, cfg in LOCAL_CONFIGS.items():
    n_tr = pd.Series([SOURCE_OF[i] for i in cfg["train"]]).value_counts().to_dict()
    n_te = pd.Series([SOURCE_OF[i] for i in cfg["test"]]).value_counts().to_dict()
    print(f"cfg{k}: train {len(cfg['train'])} {n_tr} | test {len(cfg['test'])} {n_te}")
print("split_hash =", SPLIT_HASH)

## 6. Cache
All Japanese and local images and masks are resized once so that the short side is 512 px. Local evaluation also runs at this resolution.

In [ ]:
def rgb_to_index(mask_rgb: np.ndarray) -> np.ndarray:
    dist = ((mask_rgb[:, :, None, :].astype(np.int32) - PALETTE[None, None]) ** 2).sum(-1)
    return dist.argmin(-1).astype(np.uint8)


def load_label(path: str, fmt: str) -> tuple[np.ndarray, float]:
    if fmt == "index":
        label = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        assert label is not None and label.max() <= 2, f"Invalid index mask: {path}"
        return label, 0.0
    if fmt == "local":
        return read_local_mask(path)
    return rgb_to_index(cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)), 0.0


def cache_one(row) -> dict:
    img_out, mask_out = CACHE / f"{row.id}.jpg", CACHE / f"{row.id}.png"
    img = cv2.imread(row.image_path, cv2.IMREAD_COLOR)
    assert img is not None, f"Cannot read {row.image_path}"
    h, w = img.shape[:2]
    info = {"id": row.id, "width": w, "height": h, "ambiguous": 0.0, "mask_w": w, "mask_h": h}
    cached = img_out.is_file() and mask_out.is_file()
    if cached and row.mask_format != "local":
        return info
    label, ambiguous = load_label(row.mask_path, row.mask_format)
    info.update(ambiguous=ambiguous, mask_h=label.shape[0], mask_w=label.shape[1])
    if cached:
        return info
    if label.shape != (h, w):
        label = cv2.resize(label, (w, h), interpolation=cv2.INTER_NEAREST)
    scale = SHORT_SIDE / min(h, w)
    size = (round(w * scale), round(h * scale))
    cv2.imwrite(str(img_out), cv2.resize(img, size, interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR),
                [cv2.IMWRITE_JPEG_QUALITY, 95])
    cv2.imwrite(str(mask_out), cv2.resize(label, size, interpolation=cv2.INTER_NEAREST))
    return info


all_rows = pd.concat([japan[japan.split != "test"], local], ignore_index=True)
CACHE.mkdir(parents=True, exist_ok=True)
t0 = time.time()
with ThreadPoolExecutor(max_workers=4) as pool:
    audit = pd.DataFrame(list(tqdm(pool.map(cache_one, all_rows.itertuples(index=False)), total=len(all_rows),
                                   desc="Cache")))
log(f"Cache ready: {len(audit)} pairs in {(time.time() - t0) / 60:.1f} min")

local_audit = audit[audit.id.str.startswith("local_")].copy()
local_audit["source"] = local_audit.id.map(SOURCE_OF)
local_audit["scale_mismatch"] = ((local_audit.width / local_audit.mask_w) / (local_audit.height / local_audit.mask_h) - 1).abs()
too_ambiguous = local_audit[local_audit.ambiguous > AMBIGUOUS_LIMIT]
assert too_ambiguous.empty, f"Local masks with ambiguous pixels > {AMBIGUOUS_LIMIT:.1%}: {too_ambiguous.id.tolist()}"
local_audit.to_csv(OUT / "audit_local.csv", index=False)
mismatch = (local_audit.width != local_audit.mask_w) | (local_audit.height != local_audit.mask_h)
print(f"Local masks resized to their image: {int(mismatch.sum())} | "
      f"aspect ratio off by > 1%: {local_audit.loc[local_audit.scale_mismatch > 0.01, 'id'].tolist()}")

## 7. Dataset, augmentation, and mixed sampler
`MixedSampler` always yields `STEPS_PER_EPOCH × 8` slots per epoch. For B, each slot holds a local image with probability `P_LOCAL`, otherwise a Japanese image (without replacement within a cycle). A has no local images.

In [ ]:
def train_transform() -> A.Compose:
    return A.Compose([
        A.PadIfNeeded(min_height=CROP, min_width=CROP, border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=IGNORE),
        A.RandomCrop(height=CROP, width=CROP),
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.85, 1.15), translate_percent=(-0.05, 0.05), rotate=(-7, 7),
                 border_mode=cv2.BORDER_CONSTANT, fill=0, fill_mask=IGNORE, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
        A.HueSaturationValue(hue_shift_limit=8, sat_shift_limit=20, val_shift_limit=15, p=0.3),
        A.GaussNoise(std_range=(0.02, 0.08), p=0.2),
        A.Normalize(mean=MEAN, std=STD),
    ])


def eval_transform() -> A.Compose:
    return A.Compose([A.Normalize(mean=MEAN, std=STD)])


class CachedDataset(Dataset):
    def __init__(self, ids: list[str], training: bool):
        self.ids = list(ids)
        self.transform = train_transform() if training else eval_transform()

    def __len__(self) -> int:
        return len(self.ids)

    def __getitem__(self, i):
        image = cv2.cvtColor(cv2.imread(str(CACHE / f"{self.ids[i]}.jpg")), cv2.COLOR_BGR2RGB)
        label = cv2.imread(str(CACHE / f"{self.ids[i]}.png"), cv2.IMREAD_GRAYSCALE)
        out = self.transform(image=image, mask=label)
        image = torch.from_numpy(np.ascontiguousarray(out["image"].transpose(2, 0, 1))).float()
        return image, torch.from_numpy(out["mask"].astype(np.int64)), i


class MixedSampler(Sampler):
    def __init__(self, n_main: int, n_local: int, p_local: float, seed: int):
        self.n_main, self.n_local = n_main, n_local
        self.p_local = p_local if n_local else 0.0
        self.total = STEPS_PER_EPOCH * BATCH_SIZE
        self.seed, self.epoch = seed, 0

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch

    def __len__(self) -> int:
        return self.total

    @staticmethod
    def _cycle(rng, n: int, count: int) -> np.ndarray:
        if count == 0:
            return np.empty(0, np.int64)
        return np.concatenate([rng.permutation(n) for _ in range(math.ceil(count / n))])[:count]

    def __iter__(self):
        rng = np.random.default_rng(self.seed * 10_007 + self.epoch)
        is_local = rng.random(self.total) < self.p_local
        order = np.empty(self.total, np.int64)
        order[~is_local] = self._cycle(rng, self.n_main, int((~is_local).sum()))
        order[is_local] = self._cycle(rng, self.n_local, int(is_local.sum())) + self.n_main
        return iter(order.tolist())


def seed_worker(worker_id: int) -> None:
    seed = torch.initial_seed() % 2**32
    random.seed(seed)
    np.random.seed(seed)
    torch.utils.data.get_worker_info().dataset.transform.set_random_seed(seed)

## 8. Model, loss, optimizer, and metrics
Evaluation stores a 3×3 confusion matrix per image (ignored and padded pixels removed); every metric is computed from these matrices. **mIoU litter** is the mean IoU of artificial and natural litter; 3-class mIoU is reported as a complement.

In [ ]:
def build_model(weights: str | None = ENCODER_WEIGHTS) -> nn.Module:
    return smp.Segformer(encoder_name=ENCODER, encoder_weights=weights,
                         classes=3, activation=None).to(DEVICE)


ce_loss = nn.CrossEntropyLoss(ignore_index=IGNORE)


def criterion(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    ce = ce_loss(logits, targets)
    prob = torch.softmax(logits.float(), dim=1)
    valid = (targets != IGNORE).float()
    terms = []
    for c in (1, 2):
        actual = (targets == c).float()
        if actual.sum() == 0:
            continue
        pred = prob[:, c] * valid
        terms.append(1 - (2 * (pred * actual).sum() + 1) / (pred.sum() + actual.sum() + 1))
    return ce + (torch.stack(terms).mean() if terms else ce.new_zeros(()))


def make_optimizer(model: nn.Module, total_steps: int):
    optimizer = torch.optim.AdamW(model.parameters(), lr=TUNED_PARAMS["learning_rate"],
                                   weight_decay=TUNED_PARAMS["weight_decay"])
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: (1 - min(s, total_steps) / total_steps) ** POLY_POWER)
    return optimizer, scheduler


CM_COLS = [f"cm_{g}{p}" for g in range(3) for p in range(3)]


@torch.no_grad()
def evaluate(model: nn.Module, ids: list[str], timing: bool = False, proba_path: Path | None = None) -> pd.DataFrame:
    model.eval()
    probas = {}
    loader = DataLoader(CachedDataset(ids, training=False), batch_size=1, num_workers=NUM_WORKERS)
    rows = []
    for image, label, i in loader:
        image, label = image.to(DEVICE), label.to(DEVICE)
        h, w = image.shape[-2:]
        ph, pw = (-h) % 32, (-w) % 32
        image = F.pad(image, (0, pw, 0, ph), value=0.0)
        if timing and DEVICE.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        with torch.autocast(device_type="cuda", enabled=USE_AMP):
            logits = model(image)[..., :h, :w]
            pred = logits.argmax(1)
        if proba_path is not None:
            prob = torch.softmax(logits.float(), dim=1)[0, 1:]
            probas[ids[int(i)]] = (prob * 255).round().to(torch.uint8).cpu().numpy()
        if timing and DEVICE.type == "cuda":
            torch.cuda.synchronize()
        ms = (time.perf_counter() - t0) * 1000
        valid = label != IGNORE
        cm = torch.bincount(label[valid] * 3 + pred[valid], minlength=9).cpu().numpy()
        rows.append({"id": ids[int(i)], **dict(zip(CM_COLS, cm.tolist())), "ms": ms})
    if proba_path is not None:
        np.savez_compressed(proba_path, **probas)
    return pd.DataFrame(rows)


def pooled_iou(frame: pd.DataFrame, c: int) -> float:
    tp = frame[f"cm_{c}{c}"].sum()
    gt = sum(frame[f"cm_{c}{p}"].sum() for p in range(3))
    pred = sum(frame[f"cm_{g}{c}"].sum() for g in range(3))
    return float(tp / (gt + pred - tp)) if (gt + pred - tp) else float("nan")


def val_objective(frame: pd.DataFrame) -> float:
    sub = frame[frame.source.isin(OBJECTIVE_SOURCES)]
    return float(np.mean([pooled_iou(sub, c) for c in range(3)]))


def add_image_metrics(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    valid = sum(df[c] for c in CM_COLS)
    df["valid_px"] = valid
    for c, tag in TAGS.items():
        tp = df[f"cm_{c}{c}"]
        gt = sum(df[f"cm_{c}{p}"] for p in range(3))
        pred = sum(df[f"cm_{g}{c}"] for g in range(3))
        df[f"iou_{tag}"] = (tp / (gt + pred - tp)).where(gt > 0)
        df[f"gt_pct_{tag}"] = 100 * gt / valid
        df[f"pred_pct_{tag}"] = 100 * pred / valid
    df["gt_pct_total"] = df.gt_pct_art + df.gt_pct_nat
    df["pred_pct_total"] = df.pred_pct_art + df.pred_pct_nat
    return df


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    df = add_image_metrics(df)
    rows = []
    for name, group in [("all", df)] + list(df.groupby("source")):
        ious = {c: pooled_iou(group, c) for c in range(3)}
        row = {"group": name, "n_images": len(group),
               "mIoU_litter": np.nanmean([ious[1], ious[2]]), "mIoU_3class": np.nanmean(list(ious.values())),
               "IoU_background": ious[0]}
        for c, tag in TAGS.items():
            tp = group[f"cm_{c}{c}"].sum()
            gt = sum(group[f"cm_{c}{p}"].sum() for p in range(3))
            pred = sum(group[f"cm_{g}{c}"].sum() for g in range(3))
            diff = group[f"pred_pct_{tag}"] - group[f"gt_pct_{tag}"]
            row.update({
                f"IoU_{tag}": ious[c], f"median_IoU_{tag}": group[f"iou_{tag}"].median(),
                f"precision_{tag}": tp / pred if pred else np.nan, f"recall_{tag}": tp / gt if gt else np.nan,
                f"Dice_{tag}": 2 * tp / (gt + pred) if (gt + pred) else np.nan,
                f"area_MAE_{tag}_pp": diff.abs().mean(), f"area_bias_{tag}_pp": diff.mean(),
                f"area_corr_{tag}": group[f"pred_pct_{tag}"].corr(group[f"gt_pct_{tag}"]),
            })
        total_diff = group.pred_pct_total - group.gt_pct_total
        row.update({"area_MAE_total_pp": total_diff.abs().mean(), "area_bias_total_pp": total_diff.mean()})
        rows.append(row)
    return pd.DataFrame(rows)

## 9. Training
A: no local images. B: Japanese data + `L_train`. The weights from the epoch with the best Japanese validation mIoU are kept in memory and used for the local evaluation.

In [ ]:
def train_run(local_ids: list[str], seed: int, tag: str):
    seed_all(seed)
    model = build_model()
    ids = list(train_df.id) + list(local_ids)
    dataset = CachedDataset(ids, training=True)
    sampler = MixedSampler(len(train_df), len(local_ids), P_LOCAL, seed)
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS,
                        worker_init_fn=seed_worker, pin_memory=True, drop_last=True,
                        generator=torch.Generator().manual_seed(seed))
    optimizer, scheduler = make_optimizer(model, EPOCHS * STEPS_PER_EPOCH)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    val_ids = list(val_df.id)
    val_sources = dict(zip(val_df.id, val_df.source))
    best_state, best_score, best_epoch, history = None, -1.0, 0, []
    for epoch in range(1, EPOCHS + 1):
        sampler.set_epoch(epoch)
        model.train()
        t0, loss_sum, n_local_seen = time.time(), 0.0, 0
        for images, targets, idx in loader:
            if hours() > HARD_STOP_H:
                log(f"{tag}: hard stop at {HARD_STOP_H} h; run discarded and repeated next session.")
                return None
            n_local_seen += int((idx >= len(train_df)).sum())
            images, targets = images.to(DEVICE, non_blocking=True), targets.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", enabled=USE_AMP):
                loss = criterion(model(images), targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            loss_sum += loss.item()
        val = evaluate(model, val_ids)
        val["source"] = val.id.map(val_sources)
        score = val_objective(val)
        history.append({"epoch": epoch, "loss": loss_sum / STEPS_PER_EPOCH, "val_miou": score,
                        "local_samples": n_local_seen, "minutes": (time.time() - t0) / 60})
        log(f"{tag} | epoch {epoch}/{EPOCHS} | loss {history[-1]['loss']:.4f} | val mIoU {score:.4f} | "
            f"local samples {n_local_seen} | {history[-1]['minutes']:.1f} min")
        if score > best_score:
            best_score, best_epoch = score, epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch, best_score

## 10. Runs
Per seed: A, then B for configurations 0–5. Runs with a marker in `runs/` are skipped. A new run starts only if it is expected to finish before `START_JOB_LIMIT_H`, based on the longest run so far.

In [ ]:
def job_list() -> list[dict]:
    jobs = []
    for seed in TRAIN_SEEDS:
        jobs.append({"name": f"A_s{seed}", "kind": "A", "seed": seed, "config": None})
        jobs += [{"name": f"B_s{seed}_c{c}", "kind": "B", "seed": seed, "config": c} for c in sorted(LOCAL_CONFIGS)]
    return jobs


def finished_durations() -> list[float]:
    return [json.loads(p.read_text())["hours"] for p in (OUT / "runs").glob("*.json")]


def run_job(job: dict) -> bool:
    marker = OUT / "runs" / f"{job['name']}.json"
    if marker.is_file():
        return True
    durations = finished_durations()
    estimate = max(durations) * 1.1 if durations else DEFAULT_JOB_H
    if hours() + estimate > START_JOB_LIMIT_H:
        log(f"Not enough time for {job['name']} (estimate {estimate:.2f} h). "
            "Continue in a new session with this output attached.")
        return False
    t0 = time.time()
    local_train = [] if job["kind"] == "A" else LOCAL_CONFIGS[job["config"]]["train"]
    log(f"Start {job['name']}: {len(train_df)} Japan + {len(local_train)} local")
    result = train_run(local_train, job["seed"], job["name"])
    if result is None:
        return False
    model, history, best_epoch, best_score = result
    history.to_csv(OUT / "history" / f"{job['name']}.csv", index=False)

    if job["kind"] == "A":
        eval_ids, evalset = list(local.id), "local150"
    else:
        eval_ids, evalset = LOCAL_CONFIGS[job["config"]]["test"], "ltest"
    proba_path = (OUT / "proba" / f"{job['name']}.npz"
                  if job["kind"] == "B" and job["seed"] in SAVE_PROBA_SEEDS else None)
    df = evaluate(model, eval_ids, timing=True, proba_path=proba_path)
    df = df.assign(source=df.id.map(SOURCE_OF), kind=job["kind"], seed=job["seed"],
                   config=job["config"] if job["config"] is not None else -1, evalset=evalset)
    df.to_csv(OUT / "eval" / f"{job['name']}_{evalset}.csv", index=False)

    marker.write_text(json.dumps({
        **job, "best_epoch": best_epoch, "val_miou": best_score, "hours": (time.time() - t0) / 3600,
        "ms_per_image_median": float(df.ms.median()), "split_hash": SPLIT_HASH, "params": TUNED_PARAMS,
    }, indent=2))
    s = summarize(df).set_index("group").loc["all"]
    log(f"Done {job['name']} | best epoch {best_epoch} | local mIoU litter {s.mIoU_litter:.4f} "
        f"(art {s.IoU_art:.4f}, nat {s.IoU_nat:.4f}) | {(time.time() - t0) / 60:.0f} min")
    del model
    torch.cuda.empty_cache()
    return True


if "train" in STAGES:
    jobs = job_list()
    for job in jobs:
        if not run_job(job):
            break
    done = sorted(p.stem for p in (OUT / "runs").glob("*.json"))
    log(f"Finished runs: {len(done)} | {done}")

## 11. Analysis: zero-shot, adaptation, and adaptation gain
For each local image, A and B are paired on the same seed and configuration (where the image is a test image), their confusion matrices averaged, and metrics computed. Δ = B − A with a 95% CI from a paired bootstrap (images resampled jointly for A and B); a difference is significant when the CI excludes zero.

The local splits are not grouped by photo location, so Δ is an upper bound on the gain for the same beaches rather than for an unseen site.

In [ ]:
def load_eval() -> pd.DataFrame:
    files = sorted((OUT / "eval").glob("*.csv"))
    if not files:
        raise FileNotFoundError("No evaluation results yet; run the train stage first.")
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


def paired_frames(results: pd.DataFrame, seeds=None):
    a = results[(results.kind == "A") & (results.evalset == "local150")]
    b = results[(results.kind == "B") & (results.evalset == "ltest")]
    if seeds is not None:
        a, b = a[a.seed.isin(seeds)], b[b.seed.isin(seeds)]
    a = pd.concat([a[a.id.isin(cfg["test"])].assign(config=c) for c, cfg in LOCAL_CONFIGS.items()])
    keys = ["id", "seed", "config"]
    merged = a[keys + CM_COLS].merge(b[keys + CM_COLS], on=keys, suffixes=("_A", "_B"))
    if merged.empty:
        raise ValueError("No A/B pairs with the same seed and configuration yet.")
    mean = merged.groupby("id")[[c + s for s in ("_A", "_B") for c in CM_COLS]].mean()
    frames = []
    for s in ("_A", "_B"):
        f = mean[[c + s for c in CM_COLS]].rename(columns=lambda c: c[:-2]).reset_index()
        f["source"] = f.id.map(SOURCE_OF)
        frames.append(f)
    return frames[0], frames[1], merged


def bootstrap_delta(fa: pd.DataFrame, fb: pd.DataFrame) -> pd.DataFrame:
    rng = np.random.default_rng(SPLIT_SEED)
    fa, fb = add_image_metrics(fa), add_image_metrics(fb)
    rows = []
    for name, idx in [("all", np.arange(len(fa)))] + [(s, np.flatnonzero(fa.source.to_numpy() == s))
                                                       for s in sorted(fa.source.unique())]:
        samples = rng.integers(0, len(idx), size=(N_BOOT, len(idx)))

        def pooled(frame, c, ix):
            tp = frame[f"cm_{c}{c}"].to_numpy(float)[idx]
            gt = sum(frame[f"cm_{c}{p}"].to_numpy(float) for p in range(3))[idx]
            pr = sum(frame[f"cm_{g}{c}"].to_numpy(float) for g in range(3))[idx]
            if ix is None:
                return tp.sum() / (gt.sum() + pr.sum() - tp.sum())
            t, g_, p_ = tp[ix].sum(1), gt[ix].sum(1), pr[ix].sum(1)
            return t / (g_ + p_ - t)

        def mae(frame, col_pred, col_gt, ix):
            err = (frame[col_pred] - frame[col_gt]).abs().to_numpy(float)[idx]
            return err.mean() if ix is None else err[ix].mean(1)

        metrics = {
            "IoU_art": lambda f, ix: pooled(f, 1, ix),
            "IoU_nat": lambda f, ix: pooled(f, 2, ix),
            "mIoU_litter": lambda f, ix: (pooled(f, 1, ix) + pooled(f, 2, ix)) / 2,
            "area_MAE_art_pp": lambda f, ix: mae(f, "pred_pct_art", "gt_pct_art", ix),
            "area_MAE_nat_pp": lambda f, ix: mae(f, "pred_pct_nat", "gt_pct_nat", ix),
            "area_MAE_total_pp": lambda f, ix: mae(f, "pred_pct_total", "gt_pct_total", ix),
        }
        for metric, fn in metrics.items():
            with np.errstate(all="ignore"):
                a_val, b_val = fn(fa, None), fn(fb, None)
                deltas = fn(fb, samples) - fn(fa, samples)
            lo, hi = np.nanpercentile(deltas, [2.5, 97.5])
            rows.append({"group": name, "metric": metric, "A_zero_shot": a_val, "B_adapted": b_val,
                         "delta_B_minus_A": b_val - a_val, "ci95_low": lo, "ci95_high": hi,
                         "significant": bool(lo > 0 or hi < 0)})
    return pd.DataFrame(rows)


if "analyze" in STAGES:
    results = load_eval()
    seeds_done = sorted(set(results[results.kind == "A"].seed) & set(results[results.kind == "B"].seed))
    log(f"Seeds with A/B pairs: {seeds_done}")
    frame_a, frame_b, merged = paired_frames(results)

    summary_a, summary_b = summarize(frame_a).assign(model="A_zero_shot"), summarize(frame_b).assign(model="B_adapted")
    summary = pd.concat([summary_a, summary_b], ignore_index=True)
    summary.to_csv(OUT / "summary_A_B_local.csv", index=False)
    delta = bootstrap_delta(frame_a, frame_b)
    delta.to_csv(OUT / "adaptation_gain.csv", index=False)

    per_run = []
    for (seed, cfg), g in merged.groupby(["seed", "config"]):
        row = {"seed": seed, "config": cfg}
        for s in ("_A", "_B"):
            f = g[[c + s for c in CM_COLS]].rename(columns=lambda c: c[:-2])
            row["mIoU_litter" + s] = (pooled_iou(f, 1) + pooled_iou(f, 2)) / 2
        per_run.append(row)
    per_run = pd.DataFrame(per_run)
    per_run["gain"] = per_run["mIoU_sampah_B"] - per_run["mIoU_sampah_A"]
    per_run.to_csv(OUT / "gain_per_config.csv", index=False)

    cols = ["model", "group", "n_images", "mIoU_litter", "IoU_art", "IoU_nat", "mIoU_3class",
            "Dice_art", "Dice_nat", "area_MAE_art_pp", "area_MAE_nat_pp", "area_MAE_total_pp"]
    print("Local test performance (averaged over seed × configuration pairs):")
    display(summary[cols].round(4))
    print("\nAdaptation gain (B - A) with 95% paired-bootstrap CI:")
    display(delta.round(4))
    print(f"\nmIoU litter gain per run: mean {per_run.gain.mean():.4f} ± {per_run.gain.std():.4f} "
          f"(n = {len(per_run)} seed × configuration pairs)")

    ms = results[results.kind == "A"].ms.median()
    gpu = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"
    print(f"Median inference time (local images, short side {SHORT_SIDE}, {gpu}): {ms:.1f} ms/image")

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
    for ax, metric in zip(axes, ["IoU_art", "IoU_nat", "mIoU_litter"]):
        sub = delta[delta.metric == metric].set_index("group")
        x = np.arange(len(sub))
        ax.bar(x - 0.2, sub.A_zero_shot, 0.4, label="A zero-shot")
        ax.bar(x + 0.2, sub.B_adapted, 0.4, label="B adapted")
        for xi, (name, r) in zip(x, sub.iterrows()):
            ax.text(xi, max(r.A_zero_shot, r.B_adapted) + 0.02,
                    f"Δ {r.delta_B_minus_A:+.3f}{'*' if r.significant else ''}", ha="center", fontsize=8)
        ax.set_xticks(x, sub.index)
        ax.set_title(metric)
        ax.set_ylim(0, 1)
    axes[0].legend(loc="upper left", fontsize=8)
    fig.suptitle("SegFormer MiT-B3: zero-shot vs adapted on local test data (* = 95% CI excludes 0)")
    fig.tight_layout()
    fig.savefig(OUT / "figures" / "zero_shot_vs_adaptation.png", dpi=150)
    plt.show()

## 12. Final model for deployment (optional)
Model F is trained on the Japanese training set plus **all** 150 local images (same settings as B, first seed). Reported performance still comes from section 11, because F has seen every local image.

In [ ]:
if "final" in STAGES and RUN_FINAL_MODEL:
    marker = OUT / "runs" / "F_final.json"
    if marker.is_file():
        log("Model F already exists.")
    else:
        durations = finished_durations()
        estimate = max(durations) * 1.1 if durations else DEFAULT_JOB_H
        if hours() + estimate > START_JOB_LIMIT_H:
            log("Not enough time for model F; run it in the next session.")
        else:
            t0 = time.time()
            result = train_run(list(local.id), TRAIN_SEEDS[0], "F_final")
            if result is not None:
                model, history, best_epoch, best_score = result
                history.to_csv(OUT / "history" / "F_final.csv", index=False)
                ckpt = OUT / "segformer_mit_b3_final_local.pt"
                torch.save(model.state_dict(), ckpt)
                config = {
                    "model": "SegFormer", "smp_class": "Segformer", "encoder": ENCODER, "classes": CLASSES, "palette_rgb": PALETTE.tolist(),
                    "short_side": SHORT_SIDE, "mean": MEAN, "std": STD, "pad_multiple": 32,
                    "checkpoint": ckpt.name, "params": TUNED_PARAMS, "tuning_best_trial": TUNING_BEST_TRIAL,
                    "tuning_best_val_miou": TUNING_BEST_VAL_MIOU, "best_epoch": best_epoch,
                    "train_data": f"{len(train_df)} Japan + {len(local)} local (p_local={P_LOCAL})",
                }
                (OUT / "model_config_final_local.json").write_text(json.dumps(config, indent=2))
                marker.write_text(json.dumps({"best_epoch": best_epoch, "val_miou": best_score,
                                              "hours": (time.time() - t0) / 3600}, indent=2))
                log(f"Model F saved: {ckpt.name} | best epoch {best_epoch}")

## 13. Output
`summary_A_B_local.csv` (A and B metrics per group), `adaptation_gain.csv` (Δ and 95% CI), `gain_per_config.csv`, `eval/` (per-image confusion matrices), `history/`, `runs/`, `splits/local_configs.json`, `figures/`, `proba/`, and model F.

In [ ]:
log(f"Session time: {hours():.2f} h")
for f in sorted(OUT.iterdir()):
    size = sum(p.stat().st_size for p in f.rglob("*") if p.is_file()) if f.is_dir() else f.stat().st_size
    print(f"{f.name:40s} {size / 1e6:8.2f} MB")